<a href="https://colab.research.google.com/github/ObaiMohammad/CV/blob/main/imag_latent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install scipy
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import os
import joblib


import scipy.io
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neural_network import MLPRegressor


In [3]:
directory = "/content/drive/MyDrive/Thesis/Datasets/OUT/"

# extract frequency
first_file = [f for f in os.listdir(directory) if f.endswith(".mat")][0]
freq = scipy.io.loadmat(os.path.join(directory,first_file))['frequency'].flatten()
print ("frequency range:" , freq.min(),"-", freq.max())

#Load data all the other data
data_rows = []
for filename in os.listdir(directory):
    if filename.endswith(".mat"):
        filepath = os.path.join(directory, filename)

        mat_data = scipy.io.loadmat(filepath)

        Z = mat_data['Zmean'].flatten()
        # Corrected syntax for accessing 'temperature' and 'SOC' from the structured 'ID' array
        temp = mat_data['ID'].flatten()[0]['temperature'].flatten()[0]
        soc = mat_data['ID'].flatten()[0]['SOC'].flatten()[0]



        #if temp != 58.0:
        data_rows.append({
            'SoC': soc,
            'Temperature': temp,
            'Zreal' : np.real(Z),
            'Zimag' : np.imag(Z)
          })

#Convert to DataFrame
df = pd.DataFrame(data_rows)

#Print to check
print("Overall data:")
print(df.head(10))
print(df.shape)
Zr_first = df['Zreal'].iloc[0]
print ("shape of Zreal in first row:", Zr_first.shape)





frequency range: 1.0 - 1000.0
Overall data:
  SoC  Temperature                                              Zreal  \
0  40         45.0  [0.045312051137288416, 0.04530401611328125, 0....   
1  68         25.4  [0.04853223037719726, 0.048493480682373045, 0....   
2  40         26.6  [0.0561853167215983, 0.056178873697916666, 0.0...   
3  68         45.5  [0.05093300247192383, 0.05090339533487955, 0.0...   
4  70         22.0  [0.06262793350219727, 0.06261197916666666, 0.0...   
5  49         50.0  [0.05076804860432943, 0.05074053319295247, 0.0...   
6  49         55.0  [0.05014458847045899, 0.05010858790079753, 0.0...   
7  68         26.5  [0.061404762268066404, 0.06141075134277344, 0....   
8  68         50.0  [0.04984974416097005, 0.04982718785603841, 0.0...   
9  48         25.3  [0.04975381215413411, 0.049726212819417324, 0....   

                                               Zimag  
0  [0.001052978793780009, 0.0009987467726071675, ...  
1  [0.0011864560842514036, 0.0011866639455

In [4]:
# =========================================
# 2. Build imaginary-only feature matrix + targets
# =========================================
# imag matrix: (n_samples, 100)
imag_data = np.vstack(df["Zimag"].to_numpy())  # each is 1D array of 100
soc = df["SoC"].to_numpy().reshape(-1, 1)
y = df["Temperature"].to_numpy().reshape(-1, 1)

# scale imag and soc and target
scaler_imag = StandardScaler()
X_imag_scaled = scaler_imag.fit_transform(imag_data)

scaler_soc = StandardScaler()
SOC_scaled = scaler_soc.fit_transform(soc)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)

print("X_imag_scaled:", X_imag_scaled.shape)


X_imag_scaled: (26, 100)


In [5]:
real_data = np.vstack(df["Zreal"].to_numpy())  # each is 1D array of 100


In [25]:
freq_rows = []
for i in range(26):
  freq_rows.append(real_data[i][99])

R_fm_values = np.array(freq_rows)

(26,)


In [6]:
# =========================================
# 3. Imaginary-part autoencoder
#    (smaller architecture, since input = 100)
# =========================================
class ImagAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

input_dim_imag = X_imag_scaled.shape[1]   # should be 100
latent_dim_imag = 16                      # you can try 16 / 32

imag_ae = ImagAutoencoder(input_dim_imag, latent_dim_imag)

criterion = nn.MSELoss()
optimizer = optim.Adam(imag_ae.parameters(), lr=1e-3)

# dataset + loader
X_imag_tensor = torch.tensor(X_imag_scaled, dtype=torch.float32)
ae_dataset = TensorDataset(X_imag_tensor, X_imag_tensor)
ae_loader = DataLoader(ae_dataset, batch_size=32, shuffle=True)

num_epochs = 80  # can adjust
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch_x, _ in ae_loader:
        optimizer.zero_grad()
        recon = imag_ae(batch_x)
        loss = criterion(recon, batch_x)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss/len(ae_loader):.6f}")


Epoch [1/80] | Loss: 1.007531
Epoch [2/80] | Loss: 1.005231
Epoch [3/80] | Loss: 1.003263
Epoch [4/80] | Loss: 1.000891
Epoch [5/80] | Loss: 0.997782
Epoch [6/80] | Loss: 0.993745
Epoch [7/80] | Loss: 0.988662
Epoch [8/80] | Loss: 0.982338
Epoch [9/80] | Loss: 0.974533
Epoch [10/80] | Loss: 0.964930
Epoch [11/80] | Loss: 0.953121
Epoch [12/80] | Loss: 0.938762
Epoch [13/80] | Loss: 0.921469
Epoch [14/80] | Loss: 0.900984
Epoch [15/80] | Loss: 0.877206
Epoch [16/80] | Loss: 0.849593
Epoch [17/80] | Loss: 0.817823
Epoch [18/80] | Loss: 0.781690
Epoch [19/80] | Loss: 0.741317
Epoch [20/80] | Loss: 0.697463
Epoch [21/80] | Loss: 0.651524
Epoch [22/80] | Loss: 0.606181
Epoch [23/80] | Loss: 0.565297
Epoch [24/80] | Loss: 0.533706
Epoch [25/80] | Loss: 0.514898
Epoch [26/80] | Loss: 0.505456
Epoch [27/80] | Loss: 0.493841
Epoch [28/80] | Loss: 0.470782
Epoch [29/80] | Loss: 0.435560
Epoch [30/80] | Loss: 0.393632
Epoch [31/80] | Loss: 0.351269
Epoch [32/80] | Loss: 0.312421
Epoch [33/80] | L

In [7]:
# =========================================
# 4. Encode all imaginary samples
# =========================================
imag_ae.eval()
with torch.no_grad():
    X_imag_encoded = imag_ae.encoder(X_imag_tensor).numpy()

print("Encoded imag shape:", X_imag_encoded.shape)


Encoded imag shape: (26, 16)


In [8]:
# =========================================
# 5. Build final input: (imag latent + SOC)
# =========================================
X_imag_latent = np.hstack([X_imag_encoded, SOC_scaled])
print("Imag-latent + SOC shape:", X_imag_latent.shape)


Imag-latent + SOC shape: (26, 17)


In [9]:

import joblib
# =========================================
# 6. Load MLP params and run plain K-Fold CV (no function)
# =========================================
param_path = '/content/drive/MyDrive/Thesis/Datasets/best_params_no_WN_split.pkl'
loaded_params = joblib.load(param_path)
print("Loaded parameters:", loaded_params)


Loaded parameters: {'mlpregressor__activation': 'relu', 'mlpregressor__hidden_layer_sizes': (200, 400), 'mlpregressor__learning_rate_init': np.float64(0.01000205844942958), 'mlpregressor__max_iter': 2000, 'mlpregressor__solver': 'adam'}


In [10]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_score
import numpy as np
import time


# ------------------------------------------
# 10-Fold Cross-Validation for Imag-Latent + SOC
# ------------------------------------------

# Build MLP model using the best parameters
mlp = MLPRegressor(
    hidden_layer_sizes=loaded_params["mlpregressor__hidden_layer_sizes"],
    learning_rate_init=loaded_params["mlpregressor__learning_rate_init"],
    max_iter=loaded_params["mlpregressor__max_iter"],
    activation=loaded_params["mlpregressor__activation"],
    solver=loaded_params["mlpregressor__solver"],
    random_state=42
)

start_time = time.time()

# Run 10-fold CV (negative MSE → convert later)
mse_scores = cross_val_score(
    mlp,
    X_imag_latent,
    y_scaled.ravel(),
    cv=10,
    scoring='neg_mean_squared_error'
)
elapsed_time = time.time() - start_time

# Compute R² as well
r2_scores = cross_val_score(
    mlp,
    X_imag_latent,
    y_scaled.ravel(),
    cv=10,
    scoring='r2'
)

# Convert to positive MSE
mse_scores = -mse_scores
mean_mse = mse_scores.mean()
std_mse = mse_scores.std()

# RMSE
rmse_scores = np.sqrt(mse_scores)
mean_rmse = rmse_scores.mean()
std_rmse = rmse_scores.std()

# Mean R²
mean_r2 = r2_scores.mean()

# ------------------------------------------
# Print results
# ------------------------------------------
print("\n📊 10-Fold Cross-Validation Results (Imag-Latent + SOC)")
print(f"Mean CV MSE   : {mean_mse:.6f}")
print(f"Std of MSE    : {std_mse:.6f}")
print(f"Mean CV RMSE  : {mean_rmse:.6f}")
print(f"Std of RMSE   : {std_rmse:.6f}")
print(f"Mean R²       : {mean_r2:.6f}")
print(f"Training Time : {elapsed_time:.2f} s")



📊 10-Fold Cross-Validation Results (Imag-Latent + SOC)
Mean CV MSE   : 0.047989
Std of MSE    : 0.057164
Mean CV RMSE  : 0.186370
Std of RMSE   : 0.115130
Mean R²       : 0.261636
Training Time : 5.16 s


In [11]:
results_10f_df = pd.DataFrame([{
    "Input Type": "Imag-Latent + SOC",
    "MSE": mean_mse,
    "Std_MSE": std_mse,
    "RMSE": mean_rmse,
    "Std_RMSE": std_rmse,
    "Mean_R2": mean_r2,
    "Time_sec": elapsed_time
}])

print(results_10f_df)

import os
#from google.colab import drive
#drive.mount('/content/drive')

# Path to your Thesis folder
save_dir = "/content/drive/MyDrive/Thesis"
os.makedirs(save_dir, exist_ok=True)

# Example DataFrame to save
# (replace `results_df` with your actual DataFrame name)
output_path = os.path.join(save_dir, "imag_latent_10f_results.csv")
results_10f_df.to_csv(output_path, index=False)

print(f"✅ Imag-Latent_10 results saved as CSV to: {output_path}")


          Input Type       MSE   Std_MSE     RMSE  Std_RMSE   Mean_R2  \
0  Imag-Latent + SOC  0.047989  0.057164  0.18637   0.11513  0.261636   

   Time_sec  
0  5.161594  
✅ Imag-Latent_10 results saved as CSV to: /content/drive/MyDrive/Thesis/imag_latent_10f_results.csv


In [12]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_score
import numpy as np
import time


# ------------------------------------------
# 5-Fold Cross-Validation for Imag-Latent + SOC
# ------------------------------------------

# Build MLP model using the best parameters
mlp = MLPRegressor(
    hidden_layer_sizes=loaded_params["mlpregressor__hidden_layer_sizes"],
    learning_rate_init=loaded_params["mlpregressor__learning_rate_init"],
    max_iter=loaded_params["mlpregressor__max_iter"],
    activation=loaded_params["mlpregressor__activation"],
    solver=loaded_params["mlpregressor__solver"],
    random_state=42
)

start_time = time.time()

# Run 5-fold CV (negative MSE → convert later)
mse_scores = cross_val_score(
    mlp,
    X_imag_latent,
    y_scaled.ravel(),
    cv=5,
    scoring='neg_mean_squared_error'
)
elapsed_time = time.time() - start_time

# Compute R² as well
r2_scores = cross_val_score(
    mlp,
    X_imag_latent,
    y_scaled.ravel(),
    cv=5,
    scoring='r2'
)

# Convert to positive MSE
mse_scores = -mse_scores
mean_mse = mse_scores.mean()
std_mse = mse_scores.std()

# RMSE
rmse_scores = np.sqrt(mse_scores)
mean_rmse = rmse_scores.mean()
std_rmse = rmse_scores.std()

# Mean R²
mean_r2 = r2_scores.mean()

# ------------------------------------------
# Print results
# ------------------------------------------
print("\n📊 5-Fold Cross-Validation Results (Imag-Latent + SOC)")
print(f"Mean CV MSE   : {mean_mse:.6f}")
print(f"Std of MSE    : {std_mse:.6f}")
print(f"Mean CV RMSE  : {mean_rmse:.6f}")
print(f"Std of RMSE   : {std_rmse:.6f}")
print(f"Mean R²       : {mean_r2:.6f}")
print(f"Training Time : {elapsed_time:.2f} s")



📊 5-Fold Cross-Validation Results (Imag-Latent + SOC)
Mean CV MSE   : 0.041972
Std of MSE    : 0.036133
Mean CV RMSE  : 0.186258
Std of RMSE   : 0.085325
Mean R²       : 0.956131
Training Time : 3.40 s


In [13]:
results_5f_df = pd.DataFrame([{
    "Input Type": "Imag-Latent + SOC",
    "MSE": mean_mse,
    "Std_MSE": std_mse,
    "RMSE": mean_rmse,
    "Std_RMSE": std_rmse,
    "Mean_R2": mean_r2,
    "Time_sec": elapsed_time
}])

print(results_5f_df)

import os
#from google.colab import drive
#drive.mount('/content/drive')

# Path to your Thesis folder
save_dir = "/content/drive/MyDrive/Thesis"
os.makedirs(save_dir, exist_ok=True)

# Example DataFrame to save
# (replace `results_df` with your actual DataFrame name)
output_path = os.path.join(save_dir, "imag_latent_5f_results.csv")
results_5f_df.to_csv(output_path, index=False)

print(f"✅ Imag-Latent_5 results saved as CSV to: {output_path}")


          Input Type       MSE   Std_MSE      RMSE  Std_RMSE   Mean_R2  \
0  Imag-Latent + SOC  0.041972  0.036133  0.186258  0.085325  0.956131   

   Time_sec  
0  3.401729  
✅ Imag-Latent_5 results saved as CSV to: /content/drive/MyDrive/Thesis/imag_latent_5f_results.csv


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
import time

# -----------------------------
# Model (MLP equivalent)
# -----------------------------
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 200),
            nn.ReLU(),
            nn.Linear(200, 400),
            nn.ReLU(),
            nn.Linear(400, 1)
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------
# Physics-informed loss
# -----------------------------
def physics_loss(y_pred, R_fm, E, R_const, k):
    T_physics = E / (R_const * (torch.log(R_fm) - torch.log(k)))
    return torch.mean((y_pred - T_physics) ** 2)

# -----------------------------
# Data (convert to torch)
# -----------------------------
X = torch.tensor(X_imag_scaled, dtype=torch.float32)
y = torch.tensor(y_scaled, dtype=torch.float32)

# IMPORTANT: you must provide R_fm (real impedance at fm)
R_fm_tensor = torch.tensor(R_fm_values, dtype=torch.float32).view(-1, 1)

# -----------------------------
# Constants (tune these!)
# -----------------------------
E = 0.26        # activation energy
R_const = 8.314
k = 1.0
lambda_phy = 0.1  # physics weight

# -----------------------------
# 10-Fold Cross Validation
# -----------------------------
kf = KFold(n_splits=10, shuffle=True, random_state=42)

mse_list = []
rmse_list = []
r2_list = []

start_time = time.time()

for train_idx, val_idx in kf.split(X):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    R_train, R_val = R_fm_tensor[train_idx], R_fm_tensor[val_idx]

    model = MLP(input_dim=X.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    # -----------------------------
    # Training
    # -----------------------------
    for epoch in range(2000):
        model.train()

        y_pred = model(X_train)

        data_loss = criterion(y_pred, y_train)
        phy_loss = physics_loss(y_pred, R_train, E, R_const, k)

        loss = data_loss + lambda_phy * phy_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # -----------------------------
    # Evaluation
    # -----------------------------
    model.eval()
    with torch.no_grad():
        y_val_pred = model(X_val).numpy()
        y_val_np = y_val.numpy()

    mse = mean_squared_error(y_val_np, y_val_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val_np, y_val_pred)

    mse_list.append(mse)
    rmse_list.append(rmse)
    r2_list.append(r2)

elapsed_time = time.time() - start_time

# -----------------------------
# Results
# -----------------------------
print("\n📊 10-Fold CV Results (PyTorch + Physics Loss)")
print(f"Mean MSE   : {np.mean(mse_list):.6f}")
print(f"Std MSE    : {np.std(mse_list):.6f}")
print(f"Mean RMSE  : {np.mean(rmse_list):.6f}")
print(f"Std RMSE   : {np.std(rmse_list):.6f}")
print(f"Mean R²    : {np.mean(r2_list):.6f}")
print(f"Time       : {elapsed_time:.2f} s")

NameError: name 'R_fm_values' is not defined